## Model Training

In [1]:
import os, random, pathlib
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

Device: cuda


In [2]:
TRAIN_DIR = '/kaggle/input/datasets/nafeesalmahadi/oct2026-retinal-oct2017-balanced-701515-split/OCT2026/train_balanced'
VAL_DIR   = '/kaggle/input/datasets/nafeesalmahadi/oct2026-retinal-oct2017-balanced-701515-split/OCT2026/val_balanced'
TEST_DIR  = '/kaggle/input/datasets/nafeesalmahadi/oct2026-retinal-oct2017-balanced-701515-split/OCT2026/test_balanced'

CLASSES = ['CNV', 'DME', 'DRUSEN', 'NORMAL']   # confirm exact folder names match
IMG_SIZE = 224
BATCH_SIZE = 32
DRUSEN_MULTIPLIER = 2.0
EPOCHS = 40
LAMBDA_AUX = 0.3
FOCAL_GAMMA = 2.0
DROPOUT_RATE = 0.4
LR_BACKBONE = 5e-5
LR_NEW = 5e-4
WEIGHT_DECAY = 1e-4

In [3]:
def list_files(class_dir):
    return [str(p) for p in pathlib.Path(class_dir).glob('*') if p.is_file()]

def build_file_label_list(root_dir, oversample=False):
    files, labels = [], []
    for idx, cls in enumerate(CLASSES):
        cls_files = list_files(os.path.join(root_dir, cls))
        if not cls_files:
            raise ValueError(f"No files found for class '{cls}' in {root_dir} — check folder name.")
        mult = DRUSEN_MULTIPLIER if (oversample and cls == 'DRUSEN') else 1.0
        n_repeat = int(np.floor(mult))
        rep_files = cls_files * n_repeat
        extra_frac = mult - n_repeat
        if extra_frac > 0:
            n_extra = int(len(cls_files) * extra_frac)
            rep_files += random.sample(cls_files, min(n_extra, len(cls_files)))
        files += rep_files
        labels += [idx] * len(rep_files)
    return files, labels

train_files, train_labels = build_file_label_list(TRAIN_DIR, oversample=True)
val_files, val_labels     = build_file_label_list(VAL_DIR,   oversample=False)
test_files, test_labels   = build_file_label_list(TEST_DIR,  oversample=False)

print("Train class counts:", np.bincount(train_labels))

Train class counts: [26080  8043 12146 18501]


In [4]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Lambda(lambda t: t + torch.randn_like(t) * 0.02),   # mild Gaussian noise
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class OCTDataset(Dataset):
    def __init__(self, files, labels, transform):
        self.files = files
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img = Image.open(self.files[idx]).convert('RGB')
        img = self.transform(img)
        label = self.labels[idx]
        return img, label

train_ds = OCTDataset(train_files, train_labels, train_transform)
val_ds   = OCTDataset(val_files, val_labels, eval_transform)
test_ds  = OCTDataset(test_files, test_labels, eval_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

In [5]:
class ChannelAttention(nn.Module):
    def __init__(self, channels, ratio=8):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(
            nn.Linear(channels, channels // ratio),
            nn.ReLU(inplace=True),
            nn.Linear(channels // ratio, channels)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        b, c, _, _ = x.shape
        avg_out = self.mlp(self.avg_pool(x).view(b, c))
        max_out = self.mlp(self.max_pool(x).view(b, c))
        attn = self.sigmoid(avg_out + max_out).view(b, c, 1, 1)
        return x * attn

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        attn = self.sigmoid(self.conv(torch.cat([avg_out, max_out], dim=1)))
        return x * attn

class CBAM(nn.Module):
    def __init__(self, channels, ratio=8, kernel_size=7):
        super().__init__()
        self.channel_attn = ChannelAttention(channels, ratio)
        self.spatial_attn = SpatialAttention(kernel_size)

    def forward(self, x):
        x = self.channel_attn(x)
        x = self.spatial_attn(x)
        return x

In [6]:
class CBAM_VGG16_DS(nn.Module):
    def __init__(self, num_classes=len(CLASSES), dropout_rate=DROPOUT_RATE):
        super().__init__()
        vgg = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
        features = vgg.features   # 31 layers, indices 0..30

        self.block1_2 = features[0:10]    # frozen
        self.block3_4 = features[10:24]   # fine-tune (ends after block4_pool, idx 23)
        self.block5   = features[24:31]   # fine-tune (ends after block5_pool, idx 30)

        for p in self.block1_2.parameters():
            p.requires_grad = False

        self.cbam1 = CBAM(512)   # after block4 (512 channels)
        self.cbam2 = CBAM(512)   # after block5 (512 channels)

        # Auxiliary head (Block4 + CBAM-1) — deep supervision
        self.aux_pool = nn.AdaptiveAvgPool2d(1)
        self.aux_fc = nn.Sequential(
            nn.Linear(512, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, num_classes)   # logits
        )

        # Main head (Block5 + CBAM-2) — single Dropout right before output
        self.main_pool = nn.AdaptiveAvgPool2d(1)
        self.main_fc = nn.Sequential(
            nn.Linear(512, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate),
            nn.Linear(256, num_classes)   # logits
        )

    def forward(self, x):
        x = self.block1_2(x)
        x = self.block3_4(x)
        block4_feat = self.cbam1(x)

        x = self.block5(block4_feat)
        block5_feat = self.cbam2(x)

        aux = self.aux_pool(block4_feat).flatten(1)
        aux_out = self.aux_fc(aux)

        main = self.main_pool(block5_feat).flatten(1)
        main_out = self.main_fc(main)

        return main_out, aux_out

model = CBAM_VGG16_DS().to(device)
print(model)

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:02<00:00, 244MB/s]


CBAM_VGG16_DS(
  (block1_2): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block3_4): Sequential(
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel

In [7]:
counts = np.bincount(train_labels, minlength=len(CLASSES))
print("Train counts:", dict(zip(CLASSES, counts)))
assert (counts > 0).all(), "A class has zero samples — check CLASSES against folder names."

inv_freq = counts.sum() / (len(CLASSES) * counts)
class_weights = torch.tensor(inv_freq, dtype=torch.float32, device=device)
print("Class weights:", dict(zip(CLASSES, inv_freq)))

def focal_loss(logits, targets, class_weights, gamma=FOCAL_GAMMA):
    log_probs = F.log_softmax(logits, dim=1)          # stable
    probs = log_probs.exp()
    log_pt = log_probs.gather(1, targets.unsqueeze(1)).squeeze(1)
    pt = probs.gather(1, targets.unsqueeze(1)).squeeze(1)
    sample_w = class_weights[targets]
    loss = -sample_w * (1 - pt).pow(gamma) * log_pt
    return loss.mean()

def weighted_ce_loss(logits, targets, class_weights):
    return F.cross_entropy(logits, targets, weight=class_weights)   # fused, stable

new_params = list(model.cbam1.parameters()) + list(model.cbam2.parameters()) + \
             list(model.aux_fc.parameters()) + list(model.main_fc.parameters())
backbone_params = list(model.block3_4.parameters()) + list(model.block5.parameters())

optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': LR_BACKBONE},
    {'params': new_params, 'lr': LR_NEW},
], weight_decay=WEIGHT_DECAY)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=3, min_lr=1e-6
)

Train counts: {'CNV': np.int64(26080), 'DME': np.int64(8043), 'DRUSEN': np.int64(12146), 'NORMAL': np.int64(18501)}
Class weights: {'CNV': np.float64(0.6208780674846626), 'DME': np.float64(2.0132413278627377), 'DRUSEN': np.float64(1.333154948131072), 'NORMAL': np.float64(0.8752229609210312)}


In [8]:
def run_epoch(loader, training):
    model.train() if training else model.eval()
    total_loss, correct, total = 0.0, 0, 0

    with torch.set_grad_enabled(training):
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)

            main_out, aux_out = model(imgs)
            loss_main = focal_loss(main_out, labels, class_weights)
            loss_aux = weighted_ce_loss(aux_out, labels, class_weights)
            loss = loss_main + LAMBDA_AUX * loss_aux

            if training:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            if not torch.isfinite(loss):
                raise RuntimeError("Non-finite loss encountered — stop and inspect this batch.")

            total_loss += loss.item() * imgs.size(0)
            preds = main_out.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += imgs.size(0)

    return total_loss / total, correct / total

In [9]:
best_val_acc = 0.0
patience, patience_counter = 7, 0

for epoch in range(EPOCHS):
    train_loss, train_acc = run_epoch(train_loader, training=True)
    val_loss, val_acc = run_epoch(val_loader, training=False)
    scheduler.step(val_acc)

    print(f"Epoch {epoch+1}/{EPOCHS} | "
          f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
          f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), '/kaggle/working/best_cbam_vgg16_ds.pt')
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early stopping triggered.")
            break

Epoch 1/40 | train_loss=0.1867 train_acc=0.9118 | val_loss=0.1066 val_acc=0.9547
Epoch 2/40 | train_loss=0.1106 train_acc=0.9442 | val_loss=0.0857 val_acc=0.9580
Epoch 3/40 | train_loss=0.0938 train_acc=0.9524 | val_loss=0.0761 val_acc=0.9653
Epoch 4/40 | train_loss=0.0810 train_acc=0.9585 | val_loss=0.0815 val_acc=0.9633
Epoch 5/40 | train_loss=0.0719 train_acc=0.9628 | val_loss=0.0785 val_acc=0.9611
Epoch 6/40 | train_loss=0.0641 train_acc=0.9667 | val_loss=0.0716 val_acc=0.9656
Epoch 7/40 | train_loss=0.0581 train_acc=0.9692 | val_loss=0.0919 val_acc=0.9689
Epoch 8/40 | train_loss=0.0552 train_acc=0.9722 | val_loss=0.0773 val_acc=0.9716
Epoch 9/40 | train_loss=0.0497 train_acc=0.9735 | val_loss=0.0967 val_acc=0.9683
Epoch 10/40 | train_loss=0.0471 train_acc=0.9757 | val_loss=0.0712 val_acc=0.9713
Epoch 11/40 | train_loss=0.0455 train_acc=0.9774 | val_loss=0.0793 val_acc=0.9679
Epoch 12/40 | train_loss=0.0390 train_acc=0.9799 | val_loss=0.0702 val_acc=0.9713
Epoch 13/40 | train_loss=

In [10]:
torch.save(model.state_dict(), '/kaggle/working/cbam_vgg16_ds_final.pt')